# Module 3: Clone, Modify, Deploy

Clone the Echo environment from the OpenEnv repo, modify it, test locally, and deploy to HF Spaces.

**Time:** ~25 min · **Difficulty:** Intermediate · **GPU:** Not required

> **Note:** Deployment to HF Spaces (Step 6) requires a Hugging Face account and token.
> All other steps run locally.

In [1]:
# !pip install -q -e ./OpenEnv fastmcp fastapi uvicorn

import os
import sys

repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src'), os.path.join(repo, 'envs')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")

Setup complete!


## 1. Verify the Hosted Echo Environment

First, let's confirm the hosted Echo environment works.

In [2]:
from echo_env import EchoEnv
import requests

ECHO_CANDIDATES = [
    'https://openenv-echo-env.hf.space',
    'http://localhost:8001',
]

def pick_echo_url(candidates):
    for base_url in candidates:
        try:
            health = requests.get(f"{base_url}/health", timeout=8)
            if health.status_code != 200:
                continue
            metadata = requests.get(f"{base_url}/metadata", timeout=8).json()
            name = str(metadata.get('name', '')).lower()
            if 'echo' in name:
                return base_url
        except Exception:
            pass
    return None

ECHO_URL = pick_echo_url(ECHO_CANDIDATES)
print(f'Echo URL selected: {ECHO_URL}')

if ECHO_URL is None:
    print('Hosted/local echo server not available right now. Continuing with local clone workflow.')
else:
    with EchoEnv(base_url=ECHO_URL).sync() as env:
        env.reset()
        response = env.call_tool('echo_message', message='ping')
        print(f'Sent: ping')
        print(f'Received: {response}')
        print('The standard Echo returns exactly what you send.')

/mnt/d/test/.venv-reinf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Echo URL selected: http://localhost:8001
Sent: ping
Received: gnip
The standard Echo returns exactly what you send.


## 2. Clone the Echo Environment

Clone the Space repository to get the full source code.

In [3]:
# Copy the echo_env from the cloned OpenEnv repo into a working directory
import shutil, os

src = os.path.join(os.path.abspath('OpenEnv'), 'envs', 'echo_env')
dst = 'echo-env-modified'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# Ensure server/ is a proper Python package so uvicorn can import server.app
# (relative imports inside app.py require a real package, not a namespace package)
for pkg_dir in [dst, os.path.join(dst, 'server')]:
    init_file = os.path.join(pkg_dir, '__init__.py')
    if not os.path.exists(init_file):
        open(init_file, 'w').close()

print('Copied echo_env to echo-env-modified/')
print('Created __init__.py files for proper package import')
os.listdir(dst)

Copied echo_env to echo-env-modified/
Created __init__.py files for proper package import


['client.py',
 'openenv.yaml',
 'openenv_echo_env.egg-info',
 'pyproject.toml',
 'README.md',
 'server',
 '__init__.py',
 '__pycache__']

## 3. Explore the Structure

Every OpenEnv environment follows the same layout.

In [4]:
import glob
files = sorted(glob.glob('echo-env-modified/**/*', recursive=True))
for f in files:
    if os.path.isfile(f):
        print(f)

echo-env-modified/README.md
echo-env-modified/__init__.py
echo-env-modified/__pycache__/__init__.cpython-310.pyc
echo-env-modified/__pycache__/__init__.cpython-312.pyc
echo-env-modified/__pycache__/client.cpython-310.pyc
echo-env-modified/__pycache__/client.cpython-312.pyc
echo-env-modified/client.py
echo-env-modified/openenv.yaml
echo-env-modified/openenv_echo_env.egg-info/PKG-INFO
echo-env-modified/openenv_echo_env.egg-info/SOURCES.txt
echo-env-modified/openenv_echo_env.egg-info/dependency_links.txt
echo-env-modified/openenv_echo_env.egg-info/entry_points.txt
echo-env-modified/openenv_echo_env.egg-info/requires.txt
echo-env-modified/openenv_echo_env.egg-info/top_level.txt
echo-env-modified/pyproject.toml
echo-env-modified/server/Dockerfile
echo-env-modified/server/__init__.py
echo-env-modified/server/__pycache__/__init__.cpython-310.pyc
echo-env-modified/server/__pycache__/app.cpython-310.pyc
echo-env-modified/server/__pycache__/echo_environment.cpython-310.pyc
echo-env-modified/serv

In [ ]:
# Look at the MCP tool definitions in the echo environment
env_file = 'echo-env-modified/server/echo_environment.py'
with open(env_file) as f:
    print(f.read())

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Echo Environment Implementation.

A pure MCP environment that echoes back messages sent to it.
This demonstrates how to build an MCPEnvironment with inline FastMCP tools.

All interactions happen through MCP tools:
- `echo_message(message)`: Echo back the provided message
- `echo_with_length(message)`: Echo back the message with its length

Example:
    >>> from openenv.core.env_server.mcp_types import ListToolsAction, CallToolAction
    >>> env = EchoEnvironment()
    >>> env.reset()
    >>>
    >>> # List available tools
    >>> obs = env.step(ListToolsAction())
    >>> print([t.name for t in obs.tools])  # ["echo_message", "echo_with_length"]
    >>>
    >>> # Call a tool
    >>> obs = env.step(CallToolAction(tool_name="echo_message", arguments={"message": "Hi!"}))
    >>> 

## 4. Modify the Environment

Let's make a "Reverse Echo" — instead of echoing back the message, it reverses it.

We'll modify the `step()` method in `environment.py`.

In [5]:
env_file = 'echo-env-modified/server/echo_environment.py'

with open(env_file) as f:
    content = f.read()

print('Original echo_environment.py:')
print(content)

Original echo_environment.py:
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Echo Environment Implementation.

A pure MCP environment that echoes back messages sent to it.
This demonstrates how to build an MCPEnvironment with inline FastMCP tools.

All interactions happen through MCP tools:
- `echo_message(message)`: Echo back the provided message
- `echo_with_length(message)`: Echo back the message with its length

Example:
    >>> from openenv.core.env_server.mcp_types import ListToolsAction, CallToolAction
    >>> env = EchoEnvironment()
    >>> env.reset()
    >>>
    >>> # List available tools
    >>> obs = env.step(ListToolsAction())
    >>> print([t.name for t in obs.tools])  # ["echo_message", "echo_with_length"]
    >>>
    >>> # Call a tool
    >>> obs = env.step(CallToolAction(tool_name="echo_message", arguments

In [ ]:
# Modify: make echo_message reverse the input
import re

pattern = r"(def\s+echo_message\(message:\s*str\)\s*->\s*str:\s*[\s\S]*?\n\s*)return\s+message\b"
modified, count = re.subn(pattern, r"\1return message[::-1]", content, count=1)

if count != 1:
    raise RuntimeError('Could not patch echo_message return statement safely.')

with open(env_file, 'w') as f:
    f.write(modified)

print('Modified echo_environment.py (echo_message now reverses the input):')
for line in modified.split('\n'):
    if 'def echo_message' in line or 'return message[::-1]' in line or '@mcp.tool' in line:
        print(f'  {line}')

Modified echo_environment.py (echo_message now reverses the input):
          @mcp.tool
          def echo_message(message: str) -> str:
              return message[::-1]
          @mcp.tool


## 5. Test Locally

Start the modified server and connect to it.

> In Colab, we'll start the server as a background process. Locally, you'd run `uv run server` in a separate terminal.

In [6]:
import subprocess
import time
import sys
import os

# The server app imports from openenv (installed) and envs.echo_env (in OpenEnv repo).
# We run from the echo-env-modified directory so its server/ is importable.
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([
    os.path.abspath('echo-env-modified'),
    os.path.abspath('OpenEnv'),
    os.path.abspath('OpenEnv/src'),
] + env.get('PYTHONPATH', '').split(os.pathsep))

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8001'],
    cwd='echo-env-modified',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give it time to start
time.sleep(4)
print(f'Server started (PID: {server.pid})')

# Check it's healthy
import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8001/health', timeout=5) as r:
        print(f'Health: {r.read().decode()}')
except Exception as e:
    print(f'Health check failed: {e}')
    # Print server stderr for debugging
    err = server.stderr.read1(1024).decode(errors='replace')
    if err:
        print(f'Server stderr: {err}')

Server started (PID: 6236)
Health: {"status":"healthy"}


In [7]:
# Test the modified environment
# Since this is an MCP env, we use EchoEnv.call_tool()
from echo_env import EchoEnv

with EchoEnv(base_url='http://localhost:8001').sync() as env:
    env.reset()

    test_messages = ['Hello', 'OpenEnv', 'Reverse this!']
    for msg in test_messages:
        response = env.call_tool('echo_message', message=msg)
        print(f'Sent: {msg:20s} -> Received: {response}')

Sent: Hello                -> Received: olleH
Sent: OpenEnv              -> Received: vnEnepO
Sent: Reverse this!        -> Received: !siht esreveR


In [8]:
# Clean up the server
server.terminate()
server.wait()
print("Server stopped.")

Server stopped.


## 6. Deploy to HF Spaces

Once your environment works locally, deploy it with `openenv push`.

```bash
cd echo-env-modified
openenv push --repo-id YOUR_USERNAME/reverse-echo-env
```

Your environment is now live at:
- **API:** `https://YOUR_USERNAME-reverse-echo-env.hf.space`
- **Web UI:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/web`
- **Docs:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/docs`

In [ ]:
# Uncomment and run to deploy (requires HF token)
# !cd echo-env-modified && openenv push --repo-id YOUR_USERNAME/reverse-echo-env

## 7. Connect to Your Deployed Environment

After deployment, install the client and connect:

In [ ]:
# Uncomment after deploying
# !pip install -q git+https://huggingface.co/spaces/YOUR_USERNAME/reverse-echo-env
#
# with EchoEnv(base_url="https://YOUR_USERNAME-reverse-echo-env.hf.space").sync() as env:
#     result = env.reset()
#     result = env.step(EchoAction(message="Deployed!"))
#     print(f"Response from your Space: {result.observation}")

## 8. Docker Deployment (Alternative)

You can also pull and run the Docker image locally:

```bash
# Pull from HF registry (after deploying)
docker pull registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest
docker run -d -p 8001:8000 registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest

# Or build from source
cd echo-env-modified
docker build -t reverse-echo:latest -f server/Dockerfile .
docker run -d -p 8001:8000 reverse-echo:latest
```

Connect the same way:
```python
with EchoEnv(base_url="http://localhost:8001").sync() as env:
    result = env.reset()
```

## Summary

What you did:
1. Cloned an existing environment from HF Spaces
2. Explored its structure (models, client, server)
3. Modified the environment logic (echo → reverse echo)
4. Tested locally with uvicorn
5. Deployed to HF Spaces with `openenv push`

The workflow is always: **clone → modify → test → deploy**.

**Next:** [Module 4](../module-4/README.md) — Building an environment from scratch.